# Caso de negocio clasificación de clientes

## Modelo RFM - Modelo K-Means

## Fase Importación Datos

### Importar librerías

In [ ]:
import pandas as pd
import numpy as np

# pandas: librería para el análisis de datos
# numpy: librería para el cálculo numérico

### Definir las bases de datos

In [ ]:
df_parques = pd.read_csv('usos_parques_2024_2025.csv')

In [ ]:
df_parques2 = pd.read_csv('usos_parques_2024_2025.2a.csv')

### Mostrar las bases de datos definidas (filas iniciales) 

In [ ]:
df_parques.head()

In [ ]:
df_parques2.head()

### Mostrar las bases de datos definidas (filas finales) 

In [ ]:
df_parques.tail()

In [ ]:
df_parques2.tail()

### Cargar las tablas de código

In [ ]:
sedes = pd.read_excel("codigos_sedes_producto.xlsx", sheet_name="Sedes")
productos = pd.read_excel("codigos_sedes_producto.xlsx", sheet_name="Productos")

### Mostrar los códigos cargados

In [ ]:
sedes.head()

In [ ]:
productos.head()

### Mostrar las columnas y características de mi base de datos

In [ ]:
df_parques2.info()

## Fase Limpieza de datos

### Veamos la calidad de los datos de la columna "tarifa" y "tipo_venta"

In [ ]:
print(df_parques2["tarifa"].unique())

In [ ]:
print(df_parques2["tipo_venta"].unique())

### Crear función para limpiar texto

In [ ]:
import unicodedata

# unicodedata: librería interna de Python que permite trabajar con caracteres Unicode y realizar operaciones de normalización y eliminación de acentos.

def limpiar_texto(serie):
    # Quitar los espacios sobrantes (espacios internos), normalizar mayúsculas y quitar acentos
    return(serie.astype(str).str.normalize('NFKC').str.strip().str.replace(r'\s+', '', regex=True).str.upper().str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')).replace(r'ABIERTOALPUBLICO', 'ABIERTO AL PUBLICO', regex=True)
           
           #normalize('NFKC') convierte espacios especiales a su forma canónica, eliminando caracteres redundantes o combinaciones de caracteres que representan el mismo carácter.
           #strip elimina espacios al inicio y al final
           #replace (r'\s+', '', regex=True) elimina todos los espacios internos
           #upper convierte a mayúsculas
           #normalize('NFKD') descompone caracteres acentuados en sus componentes base y marcas diacríticas.
           #encode('ascii', errors='ignore') convierte los caracteres a ASCII, ignorando aquellos que no se pueden representar en ASCII (como acentos).
           #decode('utf-8') convierte los bytes resultantes de la codificación ASCII de nuevo a una cadena de texto en formato UTF-8.

df_parques2["tarifa"] = limpiar_texto(df_parques2["tarifa"])
df_parques2["tipo_venta"] = limpiar_texto(df_parques2["tipo_venta"])

In [ ]:
print(df_parques2["tarifa"].unique())

In [ ]:
print(df_parques2["tipo_venta"].unique())

### Veamos la calidad de los datos de la columna "fechanacimiento" y "fecha"

In [ ]:
df_parques2["fechanacimiento"].head(10)

In [ ]:
df_parques2["fechanacimiento"].tail(10)

In [ ]:
df_parques2["fecha"].head(10)

In [ ]:
df_parques2["fecha"].tail(10)

### Crear función para limpiar fecha

In [ ]:
# Fecha de nacimiento: fecha de nacimiento del visitante

nacimiento=df_parques2["fechanacimiento"].astype(str).str.strip().str.replace(r'\s+', '-', regex=True)
# astype(str): convierte la columna "fechanacimiento" a tipo de datos cadena (string).
# str.strip(): elimina los espacios en blanco al inicio y al final de cada valor de la columna.
# str.replace(r'\s+', '-', regex=True): reemplaza los espacios internos (uno o más espacios consecutivos) por un guion (-) utilizando una expresión regular.

df_parques2["fechanacimiento"] = pd.to_datetime(nacimiento, format='%Y-%m-%d', errors='coerce')

In [ ]:
df_parques2["fechanacimiento"].head(10)

In [171]:
df_parques2 = pd.read_csv("usos_parques_2024_2025.2.csv")

In [172]:
df_parques2["fecha"].head(10)

0    2024-01-02
1    2024-01-02
2    2024-01-02
3    2024-01-02
4      20240102
5    2024-01-02
6    2024-01-02
7    2024-01-02
8    2024-01-02
9    2024-01-02
Name: fecha, dtype: str

In [173]:
# Fecha : fecha de la visita al parque

fecha=df_parques2["fecha"].astype(str).str.strip()
# astype(str): convierte la columna "fecha" a tipo de datos cadena (string).
# str.strip(): elimina los espacios en blanco al inicio y al final de cada valor de la columna.

# Para los formatos de 8 digitos consecutivos (YYYYMMDD), se reemplaza por el formato YYYY-MM-DD:

mask8=fecha.str.fullmatch(r'\d{8}')
# mask8: variable que almacena un booleano (True o False) para cada valor de la columna "fecha", indicando si coincide con el patrón de 8 dígitos (YYYYMMDD) utilizando una expresión regular.
# str.fullmatch(r'\d{8}'): verifica si cada valor de la columna "fecha" coincide exactamente con el patrón de 8 dígitos (YYYYMMDD) utilizando una expresión regular.

fecha=fecha.where(~mask8, fecha.str.slice(0, 4) + '-' + fecha.str.slice(4, 6) + '-' + fecha.str.slice(6, 8))
# where(mask8, ...): mantiene los valores de la columna "fecha" que cumplen con la condición de mask8 (coinciden con el patrón de 8 dígitos) y reemplaza los valores que no cumplen con el patrón por una nueva cadena formateada como "YYYY-MM-DD" utilizando slicing para extraer las partes correspondientes del valor original.
#slice(0, 4): extrae los primeros 4 caracteres (año) del valor original.
#slice(4, 6): extrae los siguientes 2 caracteres (mes) del valor original.
#slice(6, 8): extrae los últimos 2 caracteres (día) del valor original.

# Normalizar el dato con formato YYYY-MM-DD, cambiando por guiones a barras (YYYY/MM/DD): 

slash=fecha.str.contains('/', na=False)
# slash: variable que almacena un booleano (True o False) para cada valor de la columna "fecha", indicando si contiene una barra (/).
# str.contains('/', na=False): verifica si cada valor de la columna "fecha" contiene una barra (/).

fecha_final=pd.Series(pd.NaT, index=fecha.index, dtype='datetime64[ns]')
# Las que tienen "/" son MM/DD/YYYY (formato U.S.A.)
fecha_final[slash] = pd.to_datetime(fecha[slash], format='%m/%d/%Y', errors='coerce')
# Las demás son YYYY-MM-DD (formato ISO 8601)
fecha_final[~slash] = pd.to_datetime(fecha[~slash], format='%Y-%m-%d', errors='coerce')

# reemplazamos la columna "fecha" en el DataFrame original con la columna "fecha_final" que contiene los valores normalizados y convertidos a tipo de datos datetime64[ns].
df_parques2["fecha"] = fecha_final

In [175]:
df_parques2["fecha"].value_counts()

fecha
2025-06-29    1797
2024-06-30    1655
2024-01-07    1534
2025-06-22    1508
2025-06-30    1372
              ... 
2024-04-22       6
2024-11-24       6
2024-10-21       5
2025-12-14       4
2024-01-22       3
Name: count, Length: 721, dtype: int64

In [176]:
df_parques2["fecha"].head()

0   2024-01-02
1   2024-01-02
2   2024-01-02
3   2024-01-02
4   2024-01-02
Name: fecha, dtype: datetime64[ns]

In [179]:
df_parques2.head()

,code_anon,genero,fechanacimiento,fecha,hora,codsede,codproducto,valor_neto,valor_descuento,tarifa,grupo,tipo_venta
0,ID_9301202a1548,Masculino,2014-04-28,2024-01-02,9.0,721.0,24.0,4500.0,0,TA,NaN,Individual
1,ID_7f5174f88b44,Femenino,1985-11-29,2024-01-02,9.0,721.0,24.0,4500.0,0,TA,NaN,Individual
2,ID_06a5d7f57dc2,Masculino,2012-07-21,2024-01-02,13.0,723.0,24.0,4500.0,0,TA,NaN,Individual
3,ID_d7a80202670e,Masculino,1975-07-17,2024-01-02,9.0,721.0,24.0,4500.0,0,TA,NaN,Individual
4,ID_bcf0ee23d281,Masculino,1997-12-23,2024-01-02,11.0,779.0,24.0,6500.0,0,TB,NaN,Individual
